# Inference on a single image (Dataset002, false-positive reduction models)

Runs the trained nnU-Net models on **one image** and shows the result. Every configuration is supported:

| configuration | what happens |
|---|---|
| `3d_lowres` | prediction at 0.97 mm |
| `3d_fullres` | prediction at 0.59 mm |
| `3d_cascade_fullres` | the `3d_lowres` model of the same trainer is run first; its prediction is an extra input of the full resolution model |

After the nnU-Net prediction, the post-processing tuned on cross-validation (`fp_postprocessing.json` in the model
folder, see `train.docx`) is applied: presence gating (presence head models), probability threshold and minimum
lesion volume. Without that file the raw nnU-Net prediction is shown.

The code is in `/workspace/segmentation/fp_inference.py` (the same code as the command line inference).

## 1. Environment

In [ ]:
import os
import sys

# nnU-Net reads its paths when it is imported -> set them first (from /workspace/.env, unless already set)
with open('/workspace/.env') as f:
    for line in f:
        line = line.split('#')[0].strip()
        if '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"'))
sys.path.insert(0, '/workspace/segmentation')

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

from fp_inference import CONFIGURATIONS, available_folds, default_postprocessing_params, model_folder, \
    predict_single_image

print('nnUNet_results =', os.environ['nnUNet_results'])

## 2. Settings

* `IMAGE_PATH`: the image to segment (any file name, `.nii.gz`).
* `LABEL_PATH`: optional reference mask, only used to show it and compute Dice (`None` if there is none).
* `FOLDS`: `None` = all folds that have a checkpoint (ensemble), or e.g. `[0]` for one fold.

In [ ]:
IMAGE_PATH = '/workspace/Dataset002_dartmouth_bilateral_mixed/imagesTr/right_case_073_00_0000.nii.gz'
LABEL_PATH = '/workspace/Dataset002_dartmouth_bilateral_mixed/labelsTr/right_case_073_00.nii.gz'
OUTPUT_ROOT = '/workspace/segmentation/inference_single'

TRAINER = 'nnUNetTrainerPresenceHead'   # or nnUNetTrainerFPReduction, nnUNetTrainer, ...
CONFIGURATIONS_TO_RUN = ['3d_lowres', '3d_fullres', '3d_cascade_fullres']
FOLDS = None
CHECKPOINT = 'checkpoint_final.safetensors'
DEVICE = 'cuda'

## 3. Which models are available?

In [ ]:
rows = []
for c in CONFIGURATIONS:
    pp = default_postprocessing_params(TRAINER, c)
    rows.append({'configuration': c, 'trained folds': available_folds(TRAINER, c, CHECKPOINT),
                 'post-processing': None if pp is None else
                 {k: pp[k] for k in ('presence_threshold', 'threshold', 'min_volume_mm3')},
                 'model folder': model_folder(TRAINER, c)})
pd.DataFrame(rows)

## 4. Run inference

Each configuration writes to `OUTPUT_ROOT/<configuration>/`: `raw/` is the nnU-Net output (mask, probabilities,
properties), `final/` the post-processed mask. `3d_cascade_fullres` needs the `3d_lowres` model of the same trainer
with the same folds.

In [ ]:
results = {}
for c in CONFIGURATIONS_TO_RUN:
    folds = FOLDS if FOLDS is not None else available_folds(TRAINER, c, CHECKPOINT)
    if c == '3d_cascade_fullres':
        folds = [f for f in folds if f in available_folds(TRAINER, '3d_lowres', CHECKPOINT)]
    if not folds:
        print(f'{c}: no trained model, skipped')
        continue
    print(f'===== {c}, folds {folds}')
    results[c] = predict_single_image(IMAGE_PATH, os.path.join(OUTPUT_ROOT, c), c, TRAINER, folds, CHECKPOINT,
                                      DEVICE)

## 5. Results

In [ ]:
reference = sitk.GetArrayFromImage(sitk.ReadImage(LABEL_PATH)) > 0 if LABEL_PATH else None


def dice(a, b):
    return 1.0 if a.sum() + b.sum() == 0 else 2 * (a & b).sum() / (a.sum() + b.sum())


rows = []
for c, r in results.items():
    final = sitk.GetArrayFromImage(sitk.ReadImage(r['mask'])) > 0
    raw = sitk.GetArrayFromImage(sitk.ReadImage(r['raw_mask'])) > 0
    row = {'configuration': c, 'presence probability': r['presence_probability'],
           'lesion present': r['lesion_present'], 'lesion volume (mm3)': round(r['lesion_volume_mm3'])}
    if reference is not None:
        row['reference has lesion'] = bool(reference.any())
        row['Dice raw'] = round(dice(raw, reference), 3)
        row['Dice final'] = round(dice(final, reference), 3)
    rows.append(row)
pd.DataFrame(rows)

## 6. Visualization

Axial slice with the largest reference lesion area (or the largest predicted area if there is no reference
lesion). Red: reference, green: raw nnU-Net prediction, yellow: final (post-processed) prediction.

In [ ]:
image = sitk.GetArrayFromImage(sitk.ReadImage(IMAGE_PATH))
masks = {c: (sitk.GetArrayFromImage(sitk.ReadImage(r['raw_mask'])) > 0,
             sitk.GetArrayFromImage(sitk.ReadImage(r['mask'])) > 0) for c, r in results.items()}

area = reference.sum((1, 2)) if reference is not None and reference.any() else \
    sum(m[0].sum((1, 2)) for m in masks.values()) if masks else np.zeros(image.shape[0])
z = int(np.argmax(area)) if area.max() > 0 else image.shape[0] // 2
lo, hi = np.percentile(image[z], (1, 99))

fig, axes = plt.subplots(1, len(masks) + 1, figsize=(5 * (len(masks) + 1), 5), squeeze=False)
axes[0, 0].imshow(image[z], cmap='gray', vmin=lo, vmax=hi)
if reference is not None:
    axes[0, 0].contour(reference[z], [0.5], colors='red', linewidths=1)
axes[0, 0].set_title(f'image, slice {z} (red: reference)')
for ax, (c, (raw, final)) in zip(axes[0, 1:], masks.items()):
    ax.imshow(image[z], cmap='gray', vmin=lo, vmax=hi)
    for m, color in ((reference, 'red'), (raw, 'lime'), (final, 'yellow')):
        if m is not None and m[z].any():
            ax.contour(m[z], [0.5], colors=color, linewidths=1)
    p = results[c]['presence_probability']
    ax.set_title(f'{c}\npresence p = {p:.2f}' if p is not None else c)
for ax in axes[0]:
    ax.axis('off')
plt.tight_layout()
plt.show()